# Smoke Test — Verify Pipeline Works
Quick 1-epoch test to verify the entire pipeline (manifest → dataset → training → upload) works before committing to a full 40-epoch run.

**Run this first** on a new Kaggle account to catch issues early.

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))

subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "python-dotenv"], check=False)
print("Ready.")

In [ ]:
# Cell 2: Verify setup
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"HF Token: {token[:8]}...")
except Exception as e:
    print(f"ERROR: {e}")

# Check attached datasets
from pathlib import Path
input_root = Path("/kaggle/input")
attached = [d.name for d in input_root.iterdir() if d.is_dir()] if input_root.exists() else []
print(f"Attached datasets ({len(attached)}): {attached}")

In [ ]:
# Cell 3: Run 1-epoch smoke test
import os
os.environ["MFFT_MODEL_VARIANT"] = "base"

script_path = str(REPO_DIR / "kaggle_train_resumable.py")
with open(script_path) as f:
    code = f.read()

# Override for smoke test: 1 epoch, patience 1
code = code.replace('MAX_EPOCHS = 40', 'MAX_EPOCHS = 1')
code = code.replace('PATIENCE = 8', 'PATIENCE = 1')

exec(compile(code, script_path, 'exec'))

## If this passes

The pipeline works. You can now run `train_mfft_base.ipynb` for the full 40-epoch training.

## If this fails

Check:
1. All 11 datasets are attached in the Input panel
2. `HF_TOKEN` is set as a Kaggle Secret
3. Internet is enabled in notebook settings